In [10]:
# ==========================================
# Setup & Imports
# ==========================================

import re
import os
import random
import numpy as np
from collections import Counter

import torch
import torch.nn as nn

from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

from tqdm import tqdm

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using Device:", device)

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Using Device: cpu


In [11]:
# ==========================================
# Hyperparameters
# ==========================================

MAX_VOCAB_SIZE = 20000
MAX_LEN = 500

BATCH_SIZE = 64

EMBED_DIM = 200
HIDDEN_DIM = 256

LEARNING_RATE = 1e-3
EPOCHS = 10

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"

MODEL_DIR = "models"

os.makedirs(MODEL_DIR, exist_ok=True)

In [12]:
# ==========================================
# Load Dataset
# ==========================================

dataset = load_dataset("imdb")

print(dataset)

train_texts = dataset["train"]["text"]
train_labels = dataset["train"]["label"]

test_texts = dataset["test"]["text"]
test_labels = dataset["test"]["label"]

print("Train Samples:", len(train_texts))
print("Test Samples :", len(test_texts))

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})
Train Samples: 25000
Test Samples : 25000


In [13]:
# ==========================================
# Text Preprocessing
# ==========================================

def clean_text(text):
    """
    Lowercase + remove HTML tags + remove special characters
    """
    text = text.lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    return text


def tokenize(text):
    return text.split()


# ------------------------------------------
# Tokenize Training Data
# ------------------------------------------

train_tokens = [
    tokenize(clean_text(text))
    for text in tqdm(train_texts, desc="Processing Train Reviews")
]

test_tokens = [
    tokenize(clean_text(text))
    for text in tqdm(test_texts, desc="Processing Test Reviews")
]


# ------------------------------------------
# Build Vocabulary
# ------------------------------------------

counter = Counter()

for tokens in train_tokens:
    counter.update(tokens)

vocab = [PAD_TOKEN, UNK_TOKEN] + [
    word
    for word, _ in counter.most_common(MAX_VOCAB_SIZE - 2)
]

word2idx = {
    word: idx
    for idx, word in enumerate(vocab)
}

idx2word = {
    idx: word
    for word, idx in word2idx.items()
}

print(f"Vocabulary Size: {len(vocab)}")


# ------------------------------------------
# Convert Tokens → Integers
# ------------------------------------------

def numericalize(tokens):

    sequence = [
        word2idx.get(token, word2idx[UNK_TOKEN])
        for token in tokens
    ]

    # Fixed-length truncation
    sequence = sequence[:MAX_LEN]

    return sequence


train_sequences = [
    numericalize(tokens)
    for tokens in train_tokens
]

test_sequences = [
    numericalize(tokens)
    for tokens in test_tokens
]


print("Example Review Length:",
      len(train_sequences[0]))

print("Example Encoded Review:")
print(train_sequences[0][:20])

Processing Test Reviews: 100%|█████████████████████████████████████████████████| 25000/25000 [00:05<00:00, 4501.88it/s]


Vocabulary Size: 20000
Example Review Length: 280
Example Encoded Review:
[11, 1511, 11, 237, 1, 36, 58, 390, 1133, 78, 5, 31, 2, 6948, 12, 3235, 9, 52, 9, 13]


In [14]:
# ==========================================
# Dataset & DataLoader
# ==========================================

class IMDBDataset(Dataset):

    def __init__(self, sequences, labels):
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):

        return (
            torch.tensor(self.sequences[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.long)
        )


# ------------------------------------------
# Custom Collate Function
# ------------------------------------------

def collate_fn(batch):

    sequences, labels = zip(*batch)

    padded_sequences = pad_sequence(
        sequences,
        batch_first=True,
        padding_value=word2idx[PAD_TOKEN]
    )

    labels = torch.tensor(labels, dtype=torch.long)

    return padded_sequences, labels


# ------------------------------------------
# Create Datasets
# ------------------------------------------

train_dataset = IMDBDataset(
    train_sequences,
    train_labels
)

test_dataset = IMDBDataset(
    test_sequences,
    test_labels
)


# ------------------------------------------
# Create DataLoaders
# ------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)


# ------------------------------------------
# Sanity Check
# ------------------------------------------

sample_x, sample_y = next(iter(train_loader))

print("Input Shape :", sample_x.shape)
print("Label Shape :", sample_y.shape)

Input Shape : torch.Size([64, 500])
Label Shape : torch.Size([64])


In [19]:
# ==========================================
# Custom LSTM Model
# ==========================================

class SentimentLSTM(nn.Module):
    def __init__(
        self,
        vocab_size,
        embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        dropout=0.3
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=word2idx[PAD_TOKEN]
        )

        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=True  # Bidirectional
        )

        self.dropout = nn.Dropout(dropout)

        # Adjust the linear layer to take hidden_dim * 2 (due to bidirectionality)
        self.fc = nn.Linear(hidden_dim * 2, 2)

    def forward(self, x):
        x = self.embedding(x)

        # LSTM output: hidden is of shape (num_layers * num_directions, batch, hidden_dim)
        _, (hidden, _) = self.lstm(x)

        # Concatenate the final hidden states from both directions
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)

        hidden = self.dropout(hidden)

        output = self.fc(hidden)

        return output

# ==========================================
# Model Initialization
# ==========================================

model = SentimentLSTM(
    vocab_size=len(word2idx)
).to(device)

print(model)

SentimentLSTM(
  (embedding): Embedding(20000, 200, padding_idx=0)
  (lstm): LSTM(200, 256, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=512, out_features=2, bias=True)
)


In [20]:
# ==========================================
# Loss Function & Optimizer
# ==========================================

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)

# ==========================================
# Early Stopping Setup
# ==========================================

best_val_loss = float("inf")

patience = 3
counter = 0

# ==========================================
# Training Loop
# ==========================================

history = {
    "train_loss": [],
    "val_loss": [],
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": []
}

for epoch in range(EPOCHS):

    # =====================
    # TRAINING
    # =====================

    model.train()

    running_train_loss = 0

    train_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )

    for x, y in train_bar:

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        outputs = model(x)

        loss = criterion(outputs, y)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

    avg_train_loss = running_train_loss / len(train_loader)

    # =====================
    # VALIDATION
    # =====================

    model.eval()

    running_val_loss = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for x, y in test_loader:

            x = x.to(device)
            y = y.to(device)

            outputs = model(x)

            loss = criterion(outputs, y)

            running_val_loss += loss.item()

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            all_preds.extend(
                predictions.cpu().numpy()
            )

            all_labels.extend(
                y.cpu().numpy()
            )

    avg_val_loss = running_val_loss / len(test_loader)

    # =====================
    # SAVE BEST MODEL
    # =====================

    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss
        counter = 0

        torch.save(
            model.state_dict(),
            os.path.join(
                MODEL_DIR,
                "best_custom_lstm.pth"
            )
        )

        print("\n✓ Best model saved")

    else:

        counter += 1

        print(
            f"\nNo improvement ({counter}/{patience})"
        )

    # =====================
    # METRICS
    # =====================

    acc = accuracy_score(
        all_labels,
        all_preds
    )

    precision = precision_score(
        all_labels,
        all_preds,
        zero_division=0
    )

    recall = recall_score(
        all_labels,
        all_preds,
        zero_division=0
    )

    f1 = f1_score(
        all_labels,
        all_preds,
        zero_division=0
    )

    # =====================
    # STORE RESULTS
    # =====================

    history["train_loss"].append(
        avg_train_loss
    )

    history["val_loss"].append(
        avg_val_loss
    )

    history["accuracy"].append(
        acc
    )

    history["precision"].append(
        precision
    )

    history["recall"].append(
        recall
    )

    history["f1"].append(
        f1
    )

    # =====================
    # PRINT RESULTS
    # =====================

    print("\n" + "=" * 60)

    print(
        f"Epoch {epoch+1}/{EPOCHS}"
    )

    print(
        f"Train Loss : {avg_train_loss:.4f}"
    )

    print(
        f"Val Loss   : {avg_val_loss:.4f}"
    )

    print(
        f"Accuracy   : {acc:.4f}"
    )

    print(
        f"Precision  : {precision:.4f}"
    )

    print(
        f"Recall     : {recall:.4f}"
    )

    print(
        f"F1 Score   : {f1:.4f}"
    )

    print("=" * 60)

    # =====================
    # EARLY STOPPING
    # =====================

    if counter >= patience:

        print(
            "\nEarly stopping triggered."
        )

        break

print("\nTraining Complete.")
print(
    f"Best Validation Loss: {best_val_loss:.4f}"
)

Epoch 1/10: 100%|██████████████████████████████████████████████████████████████████| 391/391 [1:03:42<00:00,  9.78s/it]



✓ Best model saved

Epoch 1/10
Train Loss : 0.6494
Val Loss   : 0.6376
Accuracy   : 0.6330
Precision  : 0.6542
Recall     : 0.5642
F1 Score   : 0.6059


Epoch 2/10: 100%|████████████████████████████████████████████████████████████████████| 391/391 [35:46<00:00,  5.49s/it]



✓ Best model saved

Epoch 2/10
Train Loss : 0.5356
Val Loss   : 0.5516
Accuracy   : 0.7320
Precision  : 0.8299
Recall     : 0.5837
F1 Score   : 0.6854


Epoch 3/10: 100%|████████████████████████████████████████████████████████████████████| 391/391 [17:23<00:00,  2.67s/it]


...


No improvement (1/3)

Epoch 6/10
Train Loss : 0.1650
Val Loss   : 0.3752
Accuracy   : 0.8582
Precision  : 0.8479
Recall     : 0.8730
F1 Score   : 0.8603


Epoch 7/10: 100%|████████████████████████████████████████████████████████████████████| 391/391 [09:45<00:00,  1.50s/it]



No improvement (2/3)

Epoch 7/10
Train Loss : 0.1087
Val Loss   : 0.4398
Accuracy   : 0.8573
Precision  : 0.8381
Recall     : 0.8857
F1 Score   : 0.8612


Epoch 8/10: 100%|████████████████████████████████████████████████████████████████████| 391/391 [11:47<00:00,  1.81s/it]



No improvement (3/3)

Epoch 8/10
Train Loss : 0.0640
Val Loss   : 0.4856
Accuracy   : 0.8512
Precision  : 0.8641
Recall     : 0.8335
F1 Score   : 0.8485

Early stopping triggered.

Training Complete.
Best Validation Loss: 0.3688


In [21]:
# =====================================================
# FINAL EVALUATION + SAVE
# =====================================================

import os
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

# -----------------------------------------------------
# Create Directories
# -----------------------------------------------------

RESULTS_DIR = "results"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# -----------------------------------------------------
# Save Trained Model
# -----------------------------------------------------

model_path = os.path.join(
    MODEL_DIR,
    "custom_lstm.pth"
)

torch.save(
    model.state_dict(),
    model_path
)

print(f"Model Saved: {model_path}")

# -----------------------------------------------------
# Generate Predictions
# -----------------------------------------------------

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for x, y in test_loader:

        x = x.to(device)
        y = y.to(device)

        outputs = model(x)

        preds = torch.argmax(
            outputs,
            dim=1
        )

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_labels.extend(
            y.cpu().numpy()
        )

# -----------------------------------------------------
# Calculate Metrics
# -----------------------------------------------------

accuracy = accuracy_score(
    all_labels,
    all_preds
)

precision = precision_score(
    all_labels,
    all_preds
)

recall = recall_score(
    all_labels,
    all_preds
)

f1 = f1_score(
    all_labels,
    all_preds
)

# -----------------------------------------------------
# Save Metrics CSV
# -----------------------------------------------------

metrics_df = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],
    "Value": [
        accuracy,
        precision,
        recall,
        f1
    ]
})

metrics_path = os.path.join(
    RESULTS_DIR,
    "metrics.csv"
)

metrics_df.to_csv(
    metrics_path,
    index=False
)

print(metrics_df)

# -----------------------------------------------------
# Save Classification Report
# -----------------------------------------------------

report = classification_report(
    all_labels,
    all_preds
)

report_path = os.path.join(
    RESULTS_DIR,
    "classification_report.txt"
)

with open(report_path, "w") as f:
    f.write(report)

# -----------------------------------------------------
# Save Confusion Matrix
# -----------------------------------------------------

cm = confusion_matrix(
    all_labels,
    all_preds
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Negative", "Positive"]
)

fig, ax = plt.subplots(figsize=(6, 6))

disp.plot(ax=ax)

plt.title("Confusion Matrix")

cm_path = os.path.join(
    RESULTS_DIR,
    "confusion_matrix.png"
)

plt.savefig(
    cm_path,
    bbox_inches="tight"
)

plt.close()

# -----------------------------------------------------
# Save Loss Curves
# -----------------------------------------------------

plt.figure(figsize=(8,5))

plt.plot(
    history["train_loss"],
    label="Train Loss"
)

plt.plot(
    history["val_loss"],
    label="Validation Loss"
)

plt.title("Training vs Validation Loss")

plt.xlabel("Epoch")

plt.ylabel("Loss")

plt.legend()

loss_path = os.path.join(
    RESULTS_DIR,
    "loss_curve.png"
)

plt.savefig(
    loss_path,
    bbox_inches="tight"
)

plt.close()

# -----------------------------------------------------
# Save Accuracy Curve
# -----------------------------------------------------

plt.figure(figsize=(8,5))

plt.plot(
    history["accuracy"],
    marker="o"
)

plt.title("Accuracy Across Epochs")

plt.xlabel("Epoch")

plt.ylabel("Accuracy")

acc_path = os.path.join(
    RESULTS_DIR,
    "accuracy_curve.png"
)

plt.savefig(
    acc_path,
    bbox_inches="tight"
)

plt.close()

# -----------------------------------------------------
# Save Precision Recall F1 Curves
# -----------------------------------------------------

plt.figure(figsize=(8,5))

plt.plot(
    history["precision"],
    label="Precision"
)

plt.plot(
    history["recall"],
    label="Recall"
)

plt.plot(
    history["f1"],
    label="F1 Score"
)

plt.title("Evaluation Metrics Across Epochs")

plt.xlabel("Epoch")

plt.ylabel("Score")

plt.legend()

metrics_curve_path = os.path.join(
    RESULTS_DIR,
    "metrics_curve.png"
)

plt.savefig(
    metrics_curve_path,
    bbox_inches="tight"
)

plt.close()

# -----------------------------------------------------
# Save Training History
# -----------------------------------------------------

history_df = pd.DataFrame(history)

history_path = os.path.join(
    RESULTS_DIR,
    "training_history.csv"
)

history_df.to_csv(
    history_path,
    index=False
)

# -----------------------------------------------------
# Save Misclassified Reviews
# -----------------------------------------------------

misclassified = []

for text, actual, predicted in zip(
    test_texts,
    all_labels,
    all_preds
):

    if actual != predicted:

        misclassified.append({
            "review": text,
            "actual_label": actual,
            "predicted_label": predicted
        })

misclassified_df = pd.DataFrame(
    misclassified
)

misclassified_path = os.path.join(
    RESULTS_DIR,
    "misclassified_reviews.csv"
)

misclassified_df.to_csv(
    misclassified_path,
    index=False
)

# -----------------------------------------------------
# Save Summary File
# -----------------------------------------------------

summary_path = os.path.join(
    RESULTS_DIR,
    "summary.txt"
)

with open(summary_path, "w") as f:

    f.write("Custom LSTM Sentiment Analysis Results\n")
    f.write("="*50 + "\n\n")

    f.write(f"Accuracy : {accuracy:.4f}\n")
    f.write(f"Precision: {precision:.4f}\n")
    f.write(f"Recall   : {recall:.4f}\n")
    f.write(f"F1 Score : {f1:.4f}\n")

print("\n" + "="*60)
print("ALL FILES SAVED SUCCESSFULLY")
print("="*60)

print(f"Model                 : {model_path}")
print(f"Metrics               : {metrics_path}")
print(f"Classification Report : {report_path}")
print(f"Confusion Matrix      : {cm_path}")
print(f"Loss Curve            : {loss_path}")
print(f"Accuracy Curve        : {acc_path}")
print(f"Metrics Curve         : {metrics_curve_path}")
print(f"Training History      : {history_path}")
print(f"Misclassified Reviews : {misclassified_path}")
print(f"Summary               : {summary_path}")

Model Saved: models\custom_lstm.pth
      Metric     Value
0   Accuracy  0.851200
1  Precision  0.864074
2     Recall  0.833520
3   F1 Score  0.848522

ALL FILES SAVED SUCCESSFULLY
Model                 : models\custom_lstm.pth
Metrics               : results\metrics.csv
Classification Report : results\classification_report.txt
Confusion Matrix      : results\confusion_matrix.png
Loss Curve            : results\loss_curve.png
Accuracy Curve        : results\accuracy_curve.png
Metrics Curve         : results\metrics_curve.png
Training History      : results\training_history.csv
Misclassified Reviews : results\misclassified_reviews.csv
Summary               : results\summary.txt


In [24]:
import pandas as pd
import os

comparison_df = pd.DataFrame({
    "Model": ["Custom LSTM"],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1 Score": [f1]
})

comparison_path = os.path.join(
    RESULTS_DIR,
    "model_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False
)

print(comparison_df)

         Model  Accuracy  Precision   Recall  F1 Score
0  Custom LSTM    0.8512   0.864074  0.83352  0.848522
